# 8.6. Residual Networks \(ResNet\) and ResNeXt

[Last chapter](https://github.com/DonaldKellett/my-ascend-notebooks/blob/0ede0919c0edc08eeae1acc502e4fc33a57cc2d3/orangepiaipro-20t/13-d2l-mindspore-ch8-modern-convolutional-neural-networks/00-batch-normalization.ipynb) we saw what batch normalization is, how it works and when to use it. This time, let's study _the_ standard for CNNs in modern deep learning - ResNet.

This notebook is based on the [OrangePi AIpro \(20T\)](http://www.orangepi.org/html/hardWare/computerAndMicrocontrollers/details/Orange-Pi-AIpro%2820t%29.html) development environment providing a single Ascend 310B1 NPU chip and core. The software used in this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.12
1. MindSpore 2.9.0
1. CANN 9.0.0

In [1]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi 23.0.0                                   Version: 23.0.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          50                15    / 15            |
| 0       0                     | NA              | 0            5792 / 23673                            |
+===============================+=================+======================================================+


In [2]:
!cat requirements.txt

absl-py==2.4.0
attrs==26.1.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.7
jupyterlab-git==0.53.0
jupyter-resource-usage==1.2.1
loguru==0.7.3
matplotlib==3.10.9
mindspore==2.9.0
ml-dtypes==0.5.4
msguard==0.0.8
openpyxl==3.1.5
opentelemetry-exporter-otlp-proto-grpc==1.33.1
opentelemetry-exporter-otlp-proto-http==1.33.1
pandas~=2.2
plotly>=5.11.0
pydantic==2.13.4
sympy==1.14.0
tornado==6.5.5


In [3]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.9.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.6.2. Residual Blocks

Each residual block in ResNet is divided into 2 branches.

1. A series of convolutions, batch normalization layers and ReLU activation
    1. $3 \times 3$ convolution with `1px` padding in all directions. Optionally, the number of output channels can be different from the number of input channels. Additionally, the stride can be set larger than 1 for downsampling
    1. A batch normalization layer that comes _before_ the ReLU activation
    1. ReLU activation layer
    1. $3 \times 3$ convolution with `1px` padding in all directions. The number of channels and feature map resolution is preserved
    1. Batch normalization layer
1. An optional $1 \times 1$ convolution layer for changing the number of channels and resolution of the input image or feature map. If the number of channels and resolution remains unchanged then no layer is required and we simply use the identity mapping

The sum of outputs from both branches are computed before passing the result to the final ReLU activation layer. Note that this is _different_ from the Inception block from GoogLeNet which concatenates the output from all branches along the channel dimension.

The idea of the residual block is to allow it to more easily learn the identity mapping $f(\mathbf{x}) = \mathbf{x}$ through the second branch. This ensures that as we add more layers to our deep network, it becomes _strictly_ more expressive rather than just different, allowing it to learn better from the given training data and producing more accurate predictions.

Our implementation of the residual block from ResNet with MindSpore 2.9.0 as below.

In [5]:
import mindspore.nn as nn

class Residual(nn.Cell):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Whether to use 1x1 convolution
        self.use_1x1conv = in_channels != out_channels or stride != 1
        self.b1 = nn.SequentialCell([
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3),
            nn.BatchNorm2d(out_channels)
        ])
        b2 = []
        if self.use_1x1conv:
            b2.append(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride))
        self.b2 = nn.SequentialCell(b2)
        self.relu = nn.ReLU()

    def construct(self, X):
        return self.relu(self.b1(X) + self.b2(X))

Let's try it with and without the $1 \times 1$ convolution in the second branch, inspect the output shapes produced by our residual block.

In [6]:
import mindspore.ops as ops

residual_without_1x1conv = Residual(3, 3)
X = ops.rand((4, 3, 6, 6))
residual_without_1x1conv(X).shape

/usr/local/Ascend/cann-9.0.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:179: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/c

(4, 3, 6, 6)

In [7]:
residual_with_1x1conv = Residual(3, 6, stride=2)
residual_with_1x1conv(X).shape

(4, 6, 3, 3)

## 8.6.3. ResNet Model

ResNet comes with multiple variations depending on the total number of convolutional + FC layers. The variation we'll study this chapter is ResNet-18.

ResNet-18 is comprised of 17 convolutional layers plus a final FC layer. Like GoogLeNet, it has a clear stem/body/head structure.

The body consists of 4 modules. Each module consists of 2 residual blocks. Except for the first module, the remaining modules double the number of channels and halves the feature map resolution for downsampling.

Each module is described below in sequence. To simplify our illustration, we'll treat the stem and head as separate modules as well.

1. Stem module
    1. $7 \times 7$ convolution with a stride of 2 for downsampling. The number of channels is increased to 64
    1. Batch normalization before the ReLU activation
    1. ReLU activation layer
    1. Max pooling layer with a $3 \times 3$ pooling window and stride of 2 for downsampling
1. 1st module in body consisting of 2 residual blocks preserving the number of channels and feature map resolution
1. 2nd module in body consisting of 2 residual blocks
    1. The 1st residual block doubles the number of channels to 128 and halves the feature map resolution
    1. The 2nd residual block preserves the channel dimensionality and resolution
1. 3rd module in body similar to \(3\). The number of channels doubled to 256 and resolution halved
1. 4th module in body similar to \(4\). The number of channels doubled to 512 and resolution halved
1. Head module
    1. Global average pooling layer reducing resolution to $1 \times 1$
    1. Flattening layer
    1. Final FC layer yielding logits corresponding to class probabilities

Our implementation with MindSpore 2.9.0 below.

In [8]:
resnet18 = nn.SequentialCell([
    # Stem module
    nn.SequentialCell([
        nn.Conv2d(1, 64, kernel_size=7, stride=2),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='same')
    ]),
    # 1st module in body
    nn.SequentialCell([
        Residual(64, 64),
        Residual(64, 64)
    ]),
    # 2nd module in body
    nn.SequentialCell([
        Residual(64, 128, stride=2),
        Residual(128, 128)
    ]),
    # 3rd module in body
    nn.SequentialCell([
        Residual(128, 256, stride=2),
        Residual(256, 256)
    ]),
    # 4th module in body
    nn.SequentialCell([
        Residual(256, 512, stride=2),
        Residual(512, 512)
    ]),
    # Head module
    nn.SequentialCell([
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Dense(512, 10)
    ])
])
resnet18

SequentialCell(
  (0): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(7, 7), stride=(2, 2), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7fe984e28a0>, bias_init=None, format=NCHW)
    (1): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.1.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.1.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.1.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.1.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=SAME)
  )
  (1): SequentialCell(
    (0): Residual(
      (b1): SequentialCell(
        (0): Conv2d(input_channels=64, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dila

Let's inspect the output shape after each module given an initial grayscale input image with dimensions of $64 \times 64$ pixels.

In [9]:
import mindspore.amp as amp

resnet18_amp = amp.auto_mixed_precision(network=resnet18, amp_level='O2')
resnet18_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): SequentialCell(
      (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(7, 7), stride=(2, 2), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7fe984e28a0>, bias_init=None, format=NCHW)
      (1): _OutputTo16(
        (_backbone): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.1.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.1.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.1.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.1.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
      )
      (2): ReLU()
      (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=SAME)
    )
    (1): SequentialCell(
      (0): Residual(
        (b1): SequentialCell(
          (0): Conv2d(input_channels

In [10]:
def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 64, 64)
layer_summary(net=resnet18_amp, X_shape=X_shape)

Input shape: (1, 1, 64, 64)
Output shape from SequentialCell: (1, 10)


## 8.6.4. Training

We now train ResNet-18 on the Fashion MNIST dataset upsampled to $64 \times 64$ pixels.

In [11]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [12]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [13]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [14]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(64, 64)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [15]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [16]:
optimizer = nn.SGD(params=resnet18_amp.trainable_params(), learning_rate=0.01)
optimizer

SGD()

In [17]:
from mindspore.train import Model

model = Model(network=resnet18_amp,
              loss_fn=loss_fn,
              optimizer=optimizer,
              metrics={'accuracy', 'loss'})
model

In [18]:
from mindspore.train import EarlyStopping, LossMonitor

per_print_times = 10
callbacks = [
    EarlyStopping(patience=5, verbose=True, restore_best_weights=True),
    LossMonitor(per_print_times=per_print_times)
]
callbacks

In [19]:
epochs = 5

In [20]:
model.fit(epoch=epochs,
          train_dataset=train_ds,
          valid_dataset=test_ds,
          callbacks=callbacks)

..epoch: 1 step: 10, loss is 1.0202078819274902
epoch: 1 step: 20, loss is 0.7059301733970642
epoch: 1 step: 30, loss is 0.6993376016616821
epoch: 1 step: 40, loss is 0.5898743867874146
epoch: 1 step: 50, loss is 0.46060633659362793
epoch: 1 step: 60, loss is 0.48438477516174316
epoch: 1 step: 70, loss is 0.4652344584465027
epoch: 1 step: 80, loss is 0.47324466705322266
epoch: 1 step: 90, loss is 0.34046706557273865
epoch: 1 step: 100, loss is 0.3882179856300354
epoch: 1 step: 110, loss is 0.4178686738014221
epoch: 1 step: 120, loss is 0.30301976203918457
epoch: 1 step: 130, loss is 0.3827677369117737
epoch: 1 step: 140, loss is 0.4043889343738556
epoch: 1 step: 150, loss is 0.5317440032958984
epoch: 1 step: 160, loss is 0.4665095806121826
epoch: 1 step: 170, loss is 0.378084659576416
epoch: 1 step: 180, loss is 0.3497298061847687
epoch: 1 step: 190, loss is 0.2779167890548706
epoch: 1 step: 200, loss is 0.3511732816696167
epoch: 1 step: 210, loss is 0.4418756365776062
epoch: 1 step: 2

As seen from above, there's some overfitting for an expressive model like ResNet-18. Let's check the final validation loss and accuracy of our model.

In [21]:
metrics = model.eval(valid_dataset=test_ds)
val_acc = metrics['accuracy']
val_loss = metrics['loss']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')

Validation loss: 0.3129
Validation accuracy: 0.8951


Despite the overfitting, the final validation accuracy is still around $90\%$. Not bad!

## 8.6.5. ResNeXt

The computational complexity of a dense convolution with $c_i$ input channels and $c_o$ output channels is $\mathcal{O}(c_i \cdot c_o)$. Instead of using a single dense convolution, we split the convolution into $g$ groups of $\frac{c_i}{g}$ input channels and $\frac{c_o}{g}$ output channels and concatenate their results along the channel dimension. Then, the computational complexity is reduced to $\mathcal{O}(g \cdot \frac{c_i}{g} \cdot \frac{c_o}{g}) = \mathcal{O}(\frac{c_i \cdot c_o}{g})$ which is $g$ times faster.

Applying this optimization to the residual block from ResNet yields the corresponding block in ResNeXt. Consult [D2L](https://d2l.ai/chapter_convolutional-modern/resnet.html#resnext) for a detailed exposition of ResNeXt.

In [22]:
class ResNeXtBlock(nn.Cell):
    def __init__(self, in_channels, bot_channels, group, stride=1):
        super().__init__()
        self.b1 = nn.SequentialCell([
            nn.Conv2d(in_channels, bot_channels, kernel_size=1),
            nn.BatchNorm2d(bot_channels),
            nn.ReLU(),
            nn.Conv2d(bot_channels, bot_channels, kernel_size=3, stride=stride, group=bot_channels//group),
            nn.BatchNorm2d(bot_channels),
            nn.ReLU(),
            nn.Conv2d(bot_channels, in_channels, kernel_size=1),
            nn.BatchNorm2d(in_channels)
        ])
        self.use_1x1conv = stride != 1
        b2 = []
        if self.use_1x1conv:
            b2.append(nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=stride))
        self.b2 = nn.SequentialCell(b2)
        self.relu = nn.ReLU()

    def construct(self, X):
        return self.relu(self.b1(X) + self.b2(X))

In [23]:
blk = ResNeXtBlock(in_channels=32, bot_channels=32, group=16)
X = ops.randn(4, 32, 96, 96)
blk(X).shape

.

(4, 32, 96, 96)

## 8.6.6. Summary and Discussion

We saw in this chapter how ResNet introduced the concept of _residual connections_ which help residual blocks learn the identity function $f(\mathbf{x}) = \mathbf{x}$, thus making the resulting network strictly more expressive rather than just different. This allows deep neural networks exceeding 100 layers to be easily constructed without a drop in accuracy.

Residual connections play a key role in the [Transformer](https://en.wikipedia.org/wiki/Transformer_%28deep_learning%29) architecture in LLMs such as ChatGPT as we will see in later chapters.